In [ ]:
!pip install -U sentence-transformers

In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("punkt.zip")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import re
import nltk
import uuid
import pickle

# Download NLTK tokenizer
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/quran_hadith_data"
SAVE_DIR = "/content/drive/MyDrive/fiqhAI_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

OUTPUT_JSON = os.path.join(SAVE_DIR, "chunks2.json")
OUTPUT_PKL = os.path.join(SAVE_DIR, "chunks_metadata2.pkl")

def smart_chunk_text(text, max_words=100, overlap=20):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    word_count = 0

    for sent in sentences:
        sent_words = sent.split()
        if word_count + len(sent_words) > max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = current_chunk[-overlap:] if overlap < len(current_chunk) else current_chunk
            word_count = sum(len(s.split()) for s in current_chunk)
        current_chunk.append(sent)
        word_count += len(sent_words)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

def load_and_smart_chunk(data_dir):
    all_chunks = []
    for filename in os.listdir(data_dir):
        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(data_dir, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            content = json.load(f)

        if "quran" in filename.lower():
            for surah_name, surah_content in content.items():
                if isinstance(surah_content, list):
                    combined_text = " ".join(str(verse.get("text", "")) for verse in surah_content)
                else:
                    combined_text = str(surah_content)
                chunks = smart_chunk_text(combined_text)
                for idx, chunk in enumerate(chunks):
                    all_chunks.append({
                        "file": filename,
                        "topic": surah_name,
                        "chunk_id": f"{filename}_{idx}_{uuid.uuid4().hex[:6]}",
                        "text": chunk
                    })

        elif "hadith" in filename.lower():
            chapters = content.get("Chapters") or content.get("chapters")
            if chapters:
                for chapter in chapters:
                    chapter_name = chapter.get("Chapter_Title_English") or chapter.get("chapter") or "Unknown"
                    hadiths = chapter.get("Hadiths") or chapter.get("hadiths") or []
                    combined_text = " ".join(str(h.get("english") or h.get("text") or h) for h in hadiths)
                    chunks = smart_chunk_text(combined_text)
                    for idx, chunk in enumerate(chunks):
                        all_chunks.append({
                            "file": filename,
                            "topic": chapter_name,
                            "chunk_id": f"{filename}_{idx}_{uuid.uuid4().hex[:6]}",
                            "text": chunk
                        })

        else:
            raw_text = json.dumps(content)
            chunks = smart_chunk_text(raw_text)
            for idx, chunk in enumerate(chunks):
                all_chunks.append({
                    "file": filename,
                    "topic": "Unknown",
                    "chunk_id": f"{filename}_{idx}_{uuid.uuid4().hex[:6]}",
                    "text": chunk
                })

    return all_chunks

# 🚀 Run the chunking
smart_chunks = load_and_smart_chunk(DATA_DIR)

# ✅ Save output to Google Drive
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(smart_chunks, f, ensure_ascii=False, indent=2)

with open(OUTPUT_PKL, "wb") as f:
    pickle.dump(smart_chunks, f)

print(f"✅ Smart chunks saved to Google Drive: {len(smart_chunks)}")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
!pip install faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import pickle
import json
import os
import time  # ⏱️ For timer

# ✅ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ✅ Paths
SAVE_DIR = "/content/drive/MyDrive/fiqhAI_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)
CHUNKS_PATH = os.path.join(SAVE_DIR, "chunks2.json")
EMBEDDINGS_PATH = os.path.join(SAVE_DIR, "embeddings_partial.npy")
METADATA_PATH = os.path.join(SAVE_DIR, "chunks_metadata_partial.pkl")
INDEX_PATH = os.path.join(SAVE_DIR, "faiss_index2.index")

# ✅ Load smart chunks
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    smart_chunks = json.load(f)

# ✅ Load BGE model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# ✅ Parameters
batch_size = 128
total = len(smart_chunks)
start_index = 0

# ✅ Resume support
if os.path.exists(EMBEDDINGS_PATH):
    print("🔁 Resuming from saved embeddings...")
    existing_embeddings = np.load(EMBEDDINGS_PATH)
    start_index = existing_embeddings.shape[0]
    print(f"🔁 Already processed: {start_index}/{total}")
else:
    existing_embeddings = np.empty((0, model.get_sentence_embedding_dimension()), dtype="float32")

# ✅ Timer for autosave
last_save_time = time.time()
SAVE_INTERVAL = 30 * 60  # 30 minutes

# ✅ Process in batches
for i in range(start_index, total, batch_size):
    batch_chunks = smart_chunks[i:i+batch_size]
    texts = [chunk["text"] for chunk in batch_chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)
    embeddings = np.array(embeddings).astype("float32")

    # Append embeddings
    existing_embeddings = np.vstack((existing_embeddings, embeddings))

    # Print remaining time
    elapsed = time.time() - last_save_time
    remaining = max(0, SAVE_INTERVAL - elapsed)
    mins, secs = divmod(int(remaining), 60)
    print(f"⏳ Time left for next save: {mins:02d}:{secs:02d} | ✅ Batch: {i+batch_size}/{total}")

    # Auto-save every 30 minutes
    if elapsed > SAVE_INTERVAL:
        np.save(EMBEDDINGS_PATH, existing_embeddings)
        with open(METADATA_PATH, "wb") as f:
            pickle.dump(smart_chunks[:i+batch_size], f)
        print(f"💾 Auto-saved at batch {i+batch_size}")
        last_save_time = time.time()

# ✅ Final save
np.save(EMBEDDINGS_PATH, existing_embeddings)
with open(METADATA_PATH, "wb") as f:
    pickle.dump(smart_chunks, f)

# ✅ Create FAISS index
dimension = existing_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(existing_embeddings)
faiss.write_index(index, INDEX_PATH)

print("🎉 All embeddings and FAISS index saved to Google Drive.")